# Baseline Sound Event Detection and Localization (SELD)


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchinfo import summary
from tqdm import tqdm

from datasets import VariableLengthDataset, collate_fn
from seld_net import SELDNetBackbone
from utils import seld_evaluation_metrics

FEATURES_DIR = 'data/features_dev'
SAMPLE_RATE = 24000
FRAME_LENGTH = SAMPLE_RATE // 10
MAX_EVENTS = 5
NUM_CLASSES = 13

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [2]:
train_dataset = VariableLengthDataset(FEATURES_DIR, split='train')
test_dataset = VariableLengthDataset(FEATURES_DIR, split='test')
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

In [3]:
class SELDNet(nn.Module):
    def __init__(
        self,
        num_classes,
        num_events,
        input_dim,
        hidden_dim,
        dropout,
        rnn_layers,
        mhsa_layers,
        output,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.num_events = num_events

        if output in {'multi-task', 'multi-accddoa'}:
            self.output = output
        else:
            raise ValueError(f'Invalid output type: {output}.')

        self.backbone = SELDNetBackbone(
            num_classes,
            num_events,
            input_dim,
            hidden_dim,
            dropout,
            rnn_layers,
            mhsa_layers,
        )

        # Techniques based on: https://arxiv.org/pdf/2403.11827
        if output == 'multi-task':
            # Idea from: https://arxiv.org/pdf/2010.15306
            # Predict (x, y z) for each class
            self.doa = nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, num_classes * 3),
                nn.Tanh(),
            )
            # Predict distance for each class
            self.sde = nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, num_classes),
                nn.ReLU(),
            )
        elif output == 'multi-accddoa':
            # Idea from: https://arxiv.org/pdf/2110.07124
            # Predict (x, y, z, distance) for each track and class
            raise NotImplementedError('The multi-accddoa output is not implemented.')

            self.ddoa = nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, num_events * num_classes * 4),
            )

    def forward(self, x, lengths):
        x = self.backbone(x, lengths)
        if self.output == 'multi-task':
            doa = self.doa(x).reshape(*x.shape[:2], self.num_classes, 3)
            sde = self.sde(x).reshape(*x.shape[:2], self.num_classes, 1)
            x = torch.cat([doa, sde], dim=-1)
        elif self.output == 'multi-accddoa':
            out_shape = *x.shape[:2], self.num_events, self.num_classes, 4
            x = self.ddoa(x).reshape(*out_shape)
        return x

In [4]:
model_args = {
    'num_classes': NUM_CLASSES,
    'num_events': MAX_EVENTS,
    'input_dim': 7,
    'hidden_dim': 64,
    'dropout': 0.05,
    'rnn_layers': 2,
    'mhsa_layers': 2,
    'output': 'multi-task',
}

model = SELDNet(**model_args)
features, _, lengths = next(iter(train_dataloader))
summary(model, input_data=[features, lengths])

Layer (type:depth-idx)                   Output Shape              Param #
SELDNet                                  [8, 3284, 13, 4]          --
├─SELDNetBackbone: 1-1                   [8, 3284, 128]            --
│    └─Sequential: 2-1                   [8, 64, 2, 3284]          --
│    │    └─ConvBlock: 3-1               [8, 64, 16, 3284]         4,224
│    │    └─ConvBlock: 3-2               [8, 64, 4, 3284]          37,056
│    │    └─ConvBlock: 3-3               [8, 64, 2, 3284]          37,056
│    └─GRU: 2-2                          [15341, 128]              148,992
│    └─ModuleList: 2-5                   --                        (recursive)
│    │    └─MultiheadAttention: 3-4      [8, 3284, 128]            66,048
│    └─ModuleList: 2-6                   --                        (recursive)
│    │    └─LayerNorm: 3-5               [8, 3284, 128]            256
│    └─ModuleList: 2-5                   --                        (recursive)
│    │    └─MultiheadAttention: 3-6  

In [5]:
def spherical_to_cartesian(spherical):
    """Convert spherical tensor to cartesian coordinates."""
    azimuth_rad = torch.deg2rad(spherical[..., 0])
    elevation_rad = torch.deg2rad(spherical[..., 1])
    x = torch.cos(elevation_rad) * torch.cos(azimuth_rad)
    y = torch.cos(elevation_rad) * torch.sin(azimuth_rad)
    z = torch.sin(elevation_rad)
    return x, y, z


def cartesian_to_spherical(cartesian):
    """Convert cartesian tensor to spherical coordinates."""
    x, y, z = cartesian[..., 0], cartesian[..., 1], cartesian[..., 2]
    radius = torch.sqrt(x**2 + y**2 + z**2)
    azimuth_rad = torch.atan2(y, x)
    elevation_rad = torch.asin(z / radius)
    azimuth_deg = torch.rad2deg(azimuth_rad)
    elevation_deg = torch.rad2deg(elevation_rad)
    return azimuth_deg, elevation_deg, radius


In [6]:
def spherical_labels_to_cartesian(labels):
    """Convert polar coordinates into form suitable for ADPIT computation.
    Parameters:
        - labels: (*, 4 = class, azimuth, elevation, distance)

    Returns:
        - cartesian: (*, num classes, 4 = x, y, z, distance)
    """
    classes = labels[:, :, :, 0]
    one_hot = F.one_hot(classes, NUM_CLASSES + 1).unsqueeze(-1)
    # Class 0 is always inactive - no event
    one_hot[:, :, :, 0, 0] = 0
    x, y, z = spherical_to_cartesian(labels[:, :, :, 1:3])
    dist = labels[:, :, :, 3]
    cartesian = torch.stack([x, y, z, dist], dim=-1).unsqueeze(-2) * one_hot

    # Remove class 0
    cartesian = cartesian[:, :, :, 1:, :]
    return cartesian


def multi_accddoa_loss(outputs, labels, lengths):
    """Compute loss using MSE and ADPIT from: https://arxiv.org/pdf/2110.07124.

    Parameters:
        - outputs: (batch size, time steps, num events, num classes, 3 + 1)
        - labels: (batch size, time steps, max num events, num classes)
        - lengths: (batch size,)

    Returns:
        - loss: Tensor scalar
        TODO: get output suitable for evaluation
        - predictions: ...
        - labels: ...
    """
    # torch.Size([8, 2247, 5, 13, 4]) torch.Size([8, 2247, 5, 4])
    pass


In [7]:
def multi_task_labels(labels):
    """Convert labels to a form suitable for multi-task loss.

    Parameters:
        - labels: (*, track, 4 = class, azimuth, elevation, distance)

    Returns:
        - out: (*, num classes, 3 + 1)
    """
    x = torch.sin(labels[:, :, :, 1]) * torch.cos(labels[:, :, :, 2])
    y = torch.cos(labels[:, :, :, 1]) * torch.cos(labels[:, :, :, 2])
    z = torch.sin(labels[:, :, :, 2])
    dist = labels[:, :, :, 3].float()

    out = torch.zeros(*labels.shape[:-2], NUM_CLASSES, 3 + 1, device=device)
    classes = labels[:, :, :, 0]
    for i in range(NUM_CLASSES):
        class_mask = classes == i + 1
        # Get batch, frame and track indices for all class events
        batch_frame_track = torch.nonzero(class_mask)
        batch_frame = batch_frame_track[:, :2]

        # Remove duplicate batch and frame indices (keep minimal track)
        _, inv = torch.unique(batch_frame, return_inverse=True, dim=0)
        good_indices = torch.ones_like(inv, dtype=torch.bool)
        good_indices[1:] = inv[1:] != inv[:-1]
        batch_frame_track = batch_frame_track[good_indices]

        batches = batch_frame_track[:, 0]
        frames = batch_frame_track[:, 1]
        tracks = batch_frame_track[:, 2]
        out[batches, frames, i, 0] = x[batches, frames, tracks]
        out[batches, frames, i, 1] = y[batches, frames, tracks]
        out[batches, frames, i, 2] = z[batches, frames, tracks]
        out[batches, frames, i, 3] = dist[batches, frames, tracks]
    return out


def multi_task_predictions(outputs, lengths):
    """Create predictions from outputs or labels in multi-task format.

    Parameters:
        - outputs: (batch size, time steps, num classes, 3 + 1)
        - lengths: (batch size,)

    Returns:
        - predictions: list of np.array for each batch element, each array is
            of shape (valid time steps, 5 = frame, class, x, y, z)
    """
    predictions = []
    for i, length in enumerate(lengths):
        batch_out = outputs[i, :length]
        radius = torch.sqrt(batch_out[:, :, :3].pow(2).sum(dim=-1))
        active = radius > 0.5

        # For each frame, get active classes and their predictions
        frames_classes = torch.nonzero(active)
        frames = frames_classes[:, 0]
        classes = frames_classes[:, 1]
        x = batch_out[:, :, 0][active] / radius[active]
        y = batch_out[:, :, 1][active] / radius[active]
        z = batch_out[:, :, 2][active] / radius[active]

        predict = torch.stack([frames, classes, x, y, z], dim=-1)
        predictions.append(predict.numpy(force=True))

    return predictions


def multi_task_loss(outputs, labels, lengths, *, sde_weight=1.0):
    """Compute loss using MSE for DOA and SDE branches. The technique was
    proposed in https://arxiv.org/pdf/2403.11827 based on the idea from
    https://arxiv.org/pdf/2010.15306.

    Parameters:
        - outputs: (batch size, time steps, num classes, 3 + 1)
        - labels: (batch size, time steps, max num events, num classes)
        - lengths: (batch size,)

    Returns:
        - loss: Tensor scalar
        - predictions, labels: list [frame, class, azimuth, elevation] for
            each batch element
    """
    mt_labels = multi_task_labels(labels)
    loss = 0.0

    # Lengths are for input features, which are 5 times longer
    lengths = lengths // 5

    for i, length in enumerate(lengths):
        batch_out = outputs[i, :length]
        batch_labels = mt_labels[i, :length]
        coords = batch_out[:, :, :3]
        coords_gt = batch_labels[:, :, :3]

        # 1s for x, y, z of active events and 0s otherwise
        activity_gt = (torch.sqrt(coords_gt.pow(2).sum(dim=-1)) > 0.5).float()
        activity_gt = activity_gt.unsqueeze(-1).expand_as(coords_gt)

        # Should be 0 for inactive events (should have weight 1)
        activity = torch.sqrt(coords.pow(2).sum(dim=-1))
        activity_w = 1 - activity_gt[:, :, 0]

        act_loss = F.mse_loss(activity, activity_gt[:, :, 0], weight=activity_w)
        doa_loss = F.mse_loss(coords, coords_gt, weight=activity_gt)
        sde_loss = F.mse_loss(batch_out[:, :, 3], batch_labels[:, :, 3] / 100)
        loss += (act_loss + doa_loss + sde_weight * sde_loss) / lengths.shape[0]

    predictions = multi_task_predictions(outputs, lengths)
    labels = multi_task_predictions(mt_labels, lengths)
    return loss, predictions, labels

In [8]:
def train_model(
    model_args, train_dataloader, test_dataloader, epochs, *, sde_weight=1.0
):
    model = SELDNet(**model_args).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

    def loss_fn(*x):
        if model_args['output'] == 'multi-accddoa':
            return multi_accddoa_loss(*x)
        elif model_args['output'] == 'multi-task':
            return multi_task_loss(*x, sde_weight=sde_weight)

    for epoch in range(epochs):
        loss_sum = 0
        train_predictions, train_labels = [], []
        test_predictions, test_labels = [], []

        model.train()
        pbar = tqdm(total=len(train_dataloader), desc=f'Epoch {epoch + 1}/{epochs}')
        for j, (features, labels, lengths) in enumerate(train_dataloader):
            features = features.to(device)
            # Labels: (frame, event) -> (class, azimuth, elevation, distance)
            labels = labels.to(device)

            optimizer.zero_grad()

            # outputs: (batch size, time steps, num events, num classes)
            outputs = model(features, lengths)

            loss, predictions, labels = loss_fn(outputs, labels, lengths)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item()
            pbar.set_postfix(loss=loss_sum / (j + 1))
            pbar.update(1)

            train_predictions.extend(predictions)
            train_labels.extend(labels)

        model.eval()
        with torch.no_grad():
            test_loss = 0

            for features, labels, lengths in test_dataloader:
                features = features.to(device)
                labels = labels.to(device)
                outputs = model(features, lengths)
                loss, predictions, labels = loss_fn(outputs, labels, lengths)
                test_loss += loss.item()

                test_predictions.extend(predictions)
                test_labels.extend(labels)

            pbar.set_postfix(
                loss=loss_sum / len(train_dataloader),
                test_loss=test_loss / len(test_dataloader),
            )
            pbar.close()

        train_detected = sum(len(p) for p in train_predictions)
        train_ground_truth = sum(len(l) for l in train_labels)
        test_detected = sum(len(p) for p in test_predictions)
        test_ground_truth = sum(len(l) for l in test_labels)
        print(f'Train detected: {train_detected}, ground truth: {train_ground_truth}')
        print(f'Test detected: {test_detected}, ground truth: {test_ground_truth}')
        f1, er, le, lr = seld_evaluation_metrics(train_predictions, train_labels)
        print(f'Train F1: {f1:.2f}, ER: {er:.2f}, LE: {le:.2f}, LR: {lr:.2f}')
        f1, er, le, lr = seld_evaluation_metrics(test_predictions, test_labels)
        print(f'Test F1: {f1:.2f}, ER: {er:.2f}, LE: {le:.2f}, LR: {lr:.2f}')

        # if (epoch + 1) % 10 == 0:
        #     train_predictions = np.concatenate(train_predictions, axis=0)
        #     train_labels = np.concatenate(train_labels, axis=0)
        #     test_predictions = np.concatenate(test_predictions, axis=0)
        #     test_labels = np.concatenate(test_labels, axis=0)
        #     eval_predictions(train_predictions, train_labels)
        #     eval_predictions(test_predictions, test_labels, split='test')

    return model


In [9]:
model_args = {
    'num_classes': NUM_CLASSES,
    'num_events': MAX_EVENTS,
    'input_dim': 7,
    'hidden_dim': 64,
    'dropout': 0.05,
    'rnn_layers': 2,
    'mhsa_layers': 2,
    'output': 'multi-task',
}
train_dataset = VariableLengthDataset(
    FEATURES_DIR,
    split='train',
    augments=[2],
)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)
model = train_model(
    model_args,
    train_dataloader,
    test_dataloader,
    250,
    sde_weight=1.0,
)

Epoch 1/250: 100%|██████████| 12/12 [00:06<00:00,  1.83it/s, loss=0.795, test_loss=0.714]


Train detected: 83767, ground truth: 167199
Test detected: 0, ground truth: 144083
1132.0, 20.0, 1112.0, 18296.0, 114261.43523263931
Train F1: 0.00, ER: 1.00, LE: 95.11, LR: 0.04
0.0, 0.0, 0.0, 16302.0, 0.0
Test F1: 0.00, ER: 1.00, LE: 0.00, LR: 0.00


Epoch 2/250: 100%|██████████| 12/12 [00:06<00:00,  1.90it/s, loss=0.647, test_loss=0.674]


Train detected: 0, ground truth: 167199
Test detected: 0, ground truth: 144083
0.0, 0.0, 0.0, 19428.0, 0.0
Train F1: 0.00, ER: 1.00, LE: 0.00, LR: 0.00
0.0, 0.0, 0.0, 16302.0, 0.0
Test F1: 0.00, ER: 1.00, LE: 0.00, LR: 0.00


Epoch 3/250: 100%|██████████| 12/12 [00:06<00:00,  1.92it/s, loss=0.619, test_loss=0.656]


...

Train detected: 90883, ground truth: 167199
Test detected: 46618, ground truth: 144083
6230.0, 3178.0, 3052.0, 13198.0, 211204.10922586918
Train F1: 0.16, ER: 0.89, LE: 29.41, LR: 0.17
1701.0, 64.0, 1637.0, 14601.0, 133128.79104566574
Test F1: 0.01, ER: 1.00, LE: 96.48, LR: 0.06


Epoch 249/250: 100%|██████████| 12/12 [00:06<00:00,  1.93it/s, loss=0.256, test_loss=0.627]


Train detected: 95763, ground truth: 167199
Test detected: 44318, ground truth: 144083
6364.0, 3504.0, 2860.0, 13064.0, 205858.5294430852
Train F1: 0.17, ER: 0.88, LE: 28.76, LR: 0.17
1617.0, 78.0, 1539.0, 14685.0, 118170.29930567741
Test F1: 0.01, ER: 1.00, LE: 89.65, LR: 0.05


Epoch 250/250: 100%|██████████| 12/12 [00:05<00:00,  2.05it/s, loss=0.256, test_loss=0.644]


Train detected: 91387, ground truth: 167199
Test detected: 47533, ground truth: 144083
6165.0, 3407.0, 2758.0, 13263.0, 191581.27565984428
Train F1: 0.17, ER: 0.89, LE: 27.54, LR: 0.17
1809.0, 147.0, 1662.0, 14493.0, 134714.08279895782
Test F1: 0.01, ER: 0.99, LE: 87.14, LR: 0.06
